# Pol-ISARSpaceTarget: Oriented Bounding Box Method Comparison

Purpose:

- Compare different oriented bounding box detection methods on the previous
  Pol-ISARSpaceTarget-1.0 object-detection dataset.
- Keep dataset conversion, training settings, and evaluation protocol fixed.
- Save a clean comparison table for the guide.

Compared OBB methods:

1. YOLOv8n-OBB
2. YOLOv8s-OBB
3. YOLO11n-OBB
4. YOLO11s-OBB
5. YOLO26n-OBB, if available in the installed Ultralytics version

Main input variant:

- `single_vv`, because previous experiments showed it had the strongest mAP.

Optional input variants:

- `pauli_fusion`
- `learned_fusion`, if already generated by the previous notebook


In [ ]:
!pip -q install ultralytics pillow pyyaml pandas matplotlib opencv-python


In [ ]:
import json
import os
import re
import shutil
import time
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image
from ultralytics import YOLO

print("Ultralytics OBB docs: https://docs.ultralytics.com/tasks/obb/")


## Paths and Dataset Root


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped:", exc)

ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/drdo/Pol_ISARSpaceTarget1.0"),
    Path("/content/Pol_ISARSpaceTarget1.0"),
    Path("C:/Users/keerthish/Desktop/drdo/Pol_ISARSpaceTarget1.0"),
]
DATASET_ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), ROOT_CANDIDATES[0])
print("DATASET_ROOT:", DATASET_ROOT)

LABELTXT_DIR = DATASET_ROOT / "ImageSets" / "labelTxt"
DAT_DIR = DATASET_ROOT / "ImageSets" / "images"
TIFF_DIR = DATASET_ROOT / "TIFFImages" / "AmplitudeImages"

assert LABELTXT_DIR.exists(), f"Missing labels: {LABELTXT_DIR}"
assert DAT_DIR.exists(), f"Missing complex dat files: {DAT_DIR}"

OUT_ROOT = Path("/content/polisar_obb_yolo")
DRIVE_OUT = Path("/content/drive/MyDrive/drdo/polisar_obb_method_comparison")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)


## Label Reader and Class Summary


In [ ]:
def read_dota_label_file(path: Path) -> List[Tuple[List[float], str, int]]:
    rows = []
    if not path.exists() or not path.read_text().strip():
        return rows
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 10:
            continue
        coords = [float(x) for x in parts[:8]]
        cls_name = parts[8]
        difficult = int(float(parts[9]))
        rows.append((coords, cls_name, difficult))
    return rows


class_names = sorted(
    {
        cls
        for split in ["train", "valid", "test"]
        for txt in (LABELTXT_DIR / split).glob("*.txt")
        for _, cls, _ in read_dota_label_file(txt)
    }
)
class_to_id = {name: idx for idx, name in enumerate(class_names)}
print("Class map:", class_to_id)

split_summary = []
for split in ["train", "valid", "test"]:
    txt_files = sorted((LABELTXT_DIR / split).glob("*.txt"))
    obj_count = sum(len(read_dota_label_file(p)) for p in txt_files)
    split_summary.append({"split": split, "images": len(txt_files), "objects": obj_count})
split_summary_df = pd.DataFrame(split_summary)
display(split_summary_df)
split_summary_df.to_csv(DRIVE_OUT / "dataset_split_summary.csv", index=False)
(DRIVE_OUT / "obb_class_names.json").write_text(json.dumps(class_to_id, indent=2))


## Image Builders

The notebook rebuilds `single_vv` and `pauli_fusion` if needed. If
`learned_fusion` was generated by the previous learnable-fusion notebook, this
notebook can include it automatically.


In [ ]:
POL_SUFFIXES = ["HH", "HV", "VH", "VV"]


def robust_uint8(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    x = np.log1p(np.maximum(x, 0.0))
    lo, hi = np.percentile(x, [1, 99])
    if hi <= lo:
        hi = lo + 1e-6
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    return (255 * x).astype(np.uint8)


def base_name_candidates(base_name: str) -> List[str]:
    cleaned = re.sub(r"\s+\(\d+\)$", "", base_name)
    return [base_name, cleaned] if cleaned != base_name else [base_name]


def find_first_existing(paths: Sequence[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def read_complex_dat(path: Path, shape: Tuple[int, int] = (256, 256)) -> np.ndarray:
    raw = np.fromfile(path, dtype=np.float32)
    real = raw[0::2].reshape(shape)
    imag = raw[1::2].reshape(shape)
    return real + 1j * imag


def find_dat(base_name: str, pol: str, split: str) -> Path:
    path = find_first_existing([DAT_DIR / split / f"{candidate}_{pol}.dat" for candidate in base_name_candidates(base_name)])
    if path is None:
        raise FileNotFoundError(f"No {pol}.dat found for {base_name} in split {split}")
    return path


def build_single_vv(base_name: str, split: str) -> np.ndarray:
    tiff_path = find_first_existing([TIFF_DIR / f"{candidate}_VV.tiff" for candidate in base_name_candidates(base_name)])
    if tiff_path is not None:
        img = np.array(Image.open(tiff_path).convert("L"))
        return np.stack([img, img, img], axis=-1)
    vv = read_complex_dat(find_dat(base_name, "VV", split))
    img = robust_uint8(np.abs(vv))
    return np.stack([img, img, img], axis=-1)


def build_pauli_fusion(base_name: str, split: str) -> np.ndarray:
    c = {pol: read_complex_dat(find_dat(base_name, pol, split)) for pol in POL_SUFFIXES}
    hh, hv, vh, vv = c["HH"], c["HV"], c["VH"], c["VV"]
    r = robust_uint8(np.abs(hh - vv))
    g = robust_uint8(np.abs(hv + vh))
    b = robust_uint8(np.abs(hh + vv))
    return np.stack([r, g, b], axis=-1)


## Convert DOTA-Style OBB Labels to Ultralytics YOLO-OBB Format

Existing labels:

`x1 y1 x2 y2 x3 y3 x4 y4 class difficult`

Ultralytics OBB labels:

`class x1 y1 x2 y2 x3 y3 x4 y4`

with normalized corner coordinates in `[0, 1]`.


In [ ]:
def convert_label_to_yolo_obb(src_txt: Path, dst_txt: Path, width: int = 256, height: int = 256) -> int:
    out_lines = []
    for coords, cls_name, difficult in read_dota_label_file(src_txt):
        if difficult == 1:
            continue
        cls_id = class_to_id[cls_name]
        norm = []
        for idx, value in enumerate(coords):
            norm.append(value / width if idx % 2 == 0 else value / height)
        norm = [min(max(v, 0.0), 1.0) for v in norm]
        out_lines.append(" ".join([str(cls_id)] + [f"{v:.6f}" for v in norm]))
    dst_txt.parent.mkdir(parents=True, exist_ok=True)
    dst_txt.write_text("\n".join(out_lines) + ("\n" if out_lines else ""))
    return len(out_lines)


def write_data_yaml(out_dir: Path) -> Path:
    data_yaml = {
        "path": str(out_dir),
        "train": "images/train",
        "val": "images/valid",
        "test": "images/test",
        "names": {idx: name for idx, name in enumerate(class_names)},
    }
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))
    return yaml_path


def yolo_dataset_is_complete(out_dir: Path) -> bool:
    yaml_path = out_dir / "data.yaml"
    return (
        yaml_path.exists()
        and (out_dir / "images" / "train").exists()
        and any((out_dir / "images" / "train").glob("*.png"))
        and (out_dir / "labels" / "train").exists()
        and any((out_dir / "labels" / "train").glob("*.txt"))
    )


def build_yolo_dataset(variant: str, max_per_split: int | None = None, force_rebuild: bool = False) -> Path:
    out_dir = OUT_ROOT / variant
    if yolo_dataset_is_complete(out_dir) and not force_rebuild:
        print("Using existing dataset:", out_dir)
        return out_dir / "data.yaml"

    if out_dir.exists():
        shutil.rmtree(out_dir)
    for split in ["train", "valid", "test"]:
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        txt_files = sorted((LABELTXT_DIR / split).glob("*.txt"))
        if max_per_split is not None:
            txt_files = txt_files[:max_per_split]

        for i, txt in enumerate(txt_files, start=1):
            base = txt.stem
            if variant == "single_vv":
                img = build_single_vv(base, split)
            elif variant == "pauli_fusion":
                img = build_pauli_fusion(base, split)
            else:
                raise ValueError(f"Cannot rebuild unknown variant: {variant}")

            cv2.imwrite(str(out_dir / "images" / split / f"{base}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            convert_label_to_yolo_obb(txt, out_dir / "labels" / split / f"{base}.txt")
            if i % 100 == 0:
                print(variant, split, i, "/", len(txt_files))

    yaml_path = write_data_yaml(out_dir)
    print("Prepared:", out_dir)
    return yaml_path


## Build or Reuse Datasets


In [ ]:
MAX_PER_SPLIT = None
FORCE_REBUILD_DATASETS = False

dataset_yamls = {
    "single_vv": build_yolo_dataset("single_vv", max_per_split=MAX_PER_SPLIT, force_rebuild=FORCE_REBUILD_DATASETS),
    "pauli_fusion": build_yolo_dataset("pauli_fusion", max_per_split=MAX_PER_SPLIT, force_rebuild=FORCE_REBUILD_DATASETS),
}

learned_dir = OUT_ROOT / "learned_fusion"
if yolo_dataset_is_complete(learned_dir):
    dataset_yamls["learned_fusion"] = learned_dir / "data.yaml"
    print("Found existing learned_fusion dataset:", learned_dir)
else:
    print("learned_fusion dataset not found. Run the learnable-fusion notebook first if needed.")

print(dataset_yamls)


## OBB Method List

Official Ultralytics OBB models use the `-obb.pt` suffix and are pretrained on
DOTA-style oriented boxes. If one model is unavailable in your installed
version, this notebook logs the failure and continues with the rest.


In [ ]:
OBB_METHODS = [
    {"method": "YOLOv8n-OBB", "weights": "yolov8n-obb.pt"},
    {"method": "YOLOv8s-OBB", "weights": "yolov8s-obb.pt"},
    {"method": "YOLO11n-OBB", "weights": "yolo11n-obb.pt"},
    {"method": "YOLO11s-OBB", "weights": "yolo11s-obb.pt"},
    {"method": "YOLO26n-OBB", "weights": "yolo26n-obb.pt"},
]

# Main comparison: compare OBB methods on strongest previous input.
INPUT_VARIANTS_TO_RUN = ["single_vv"]

# Optional: uncomment to run the full grid. This takes longer.
# INPUT_VARIANTS_TO_RUN = ["single_vv", "pauli_fusion", "learned_fusion"]

EPOCHS = 50
IMGSZ = 256
BATCH = 16
PATIENCE = 15

print("Methods:", OBB_METHODS)
print("Input variants:", INPUT_VARIANTS_TO_RUN)


## Train and Evaluate One Method


In [ ]:
def metric_value(metrics, *names):
    for name in names:
        obj = metrics
        ok = True
        for part in name.split("."):
            if hasattr(obj, part):
                obj = getattr(obj, part)
            else:
                ok = False
                break
        if ok:
            try:
                return float(obj)
            except Exception:
                pass
    return None


def train_eval_obb(method_name: str, weights: str, variant: str, yaml_path: Path) -> Dict:
    run_name = f"{method_name}_{variant}".replace("/", "_").replace(" ", "_")
    started = time.time()
    row = {
        "method": method_name,
        "weights": weights,
        "input_variant": variant,
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "status": "started",
        "run_name": run_name,
    }
    try:
        model = YOLO(weights)
        model.train(
            data=str(yaml_path),
            task="obb",
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            project=str(DRIVE_OUT / "runs_obb_methods"),
            name=run_name,
            exist_ok=True,
            patience=PATIENCE,
            plots=True,
        )
        metrics = model.val(data=str(yaml_path), split="test", task="obb", imgsz=IMGSZ, plots=True)
        elapsed = time.time() - started
        row.update(
            {
                "status": "ok",
                "test_precision": metric_value(metrics, "box.mp"),
                "test_recall": metric_value(metrics, "box.mr"),
                "test_mAP50": metric_value(metrics, "box.map50"),
                "test_mAP50_95": metric_value(metrics, "box.map"),
                "elapsed_min": elapsed / 60,
                "run_dir": str(DRIVE_OUT / "runs_obb_methods" / run_name),
            }
        )
    except Exception as exc:
        row.update({"status": "failed", "error": repr(exc), "elapsed_min": (time.time() - started) / 60})
        print("FAILED:", method_name, variant, exc)
    return row


## Run Comparison

This may take several hours if all methods are run for 50 epochs. For a quick
smoke test, set `EPOCHS = 2` and `MAX_PER_SPLIT = 50`, then restart runtime and
rerun from the top.


In [ ]:
results = []
for variant in INPUT_VARIANTS_TO_RUN:
    if variant not in dataset_yamls:
        print("Skipping missing variant:", variant)
        continue
    for method in OBB_METHODS:
        row = train_eval_obb(method["method"], method["weights"], variant, dataset_yamls[variant])
        results.append(row)
        pd.DataFrame(results).to_csv(DRIVE_OUT / "obb_method_comparison_live.csv", index=False)
        print(json.dumps(row, indent=2))

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv(DRIVE_OUT / "obb_method_comparison_final.csv", index=False)
(DRIVE_OUT / "obb_method_comparison_final.json").write_text(json.dumps(results, indent=2))


## Plot Comparison


In [ ]:
ok_df = results_df[results_df["status"] == "ok"].copy()
if len(ok_df):
    ok_df = ok_df.sort_values("test_mAP50_95", ascending=False)
    display(ok_df[["method", "input_variant", "test_precision", "test_recall", "test_mAP50", "test_mAP50_95", "elapsed_min"]])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(ok_df["method"] + "\n" + ok_df["input_variant"], ok_df["test_mAP50"])
    axes[0].set_title("Test mAP50")
    axes[0].tick_params(axis="x", labelrotation=45)
    axes[0].set_ylim(0, 1)

    axes[1].bar(ok_df["method"] + "\n" + ok_df["input_variant"], ok_df["test_mAP50_95"])
    axes[1].set_title("Test mAP50-95")
    axes[1].tick_params(axis="x", labelrotation=45)
    axes[1].set_ylim(0, 1)

    fig.tight_layout()
    fig.savefig(DRIVE_OUT / "obb_method_comparison_plot.png", dpi=160)
    plt.show()
else:
    print("No completed runs to plot yet.")


## Baseline Reference From Earlier Experiments


In [ ]:
reference = pd.DataFrame(
    [
        {
            "method": "YOLO11n-OBB",
            "input_variant": "single_vv",
            "test_precision": 0.804,
            "test_recall": 0.819,
            "test_mAP50": 0.821,
            "test_mAP50_95": 0.582,
            "source": "previous completed notebook",
        },
        {
            "method": "YOLO11n-OBB",
            "input_variant": "pauli_fusion",
            "test_precision": 0.828,
            "test_recall": 0.739,
            "test_mAP50": 0.783,
            "test_mAP50_95": 0.573,
            "source": "previous completed notebook",
        },
        {
            "method": "YOLO11n-OBB",
            "input_variant": "learned_fusion",
            "test_precision": 0.770,
            "test_recall": 0.683,
            "test_mAP50": 0.743,
            "test_mAP50_95": 0.518,
            "source": "previous completed notebook",
        },
    ]
)
display(reference)
reference.to_csv(DRIVE_OUT / "previous_obb_reference_results.csv", index=False)
